# Next day: the return and the win, draw or loss, with LightGBM, a GRU and an MLP

The method of [`docs/ml_plan.md`](../docs/ml_plan.md) run end to end, on
[`src/tools/ml_models/`](../src/tools/ml_models/). Two models, each forecasting one thing about
tomorrow (the owner's choice):

- **the return model**: tomorrow's return as a point forecast (the median) and its 68% and 90%
  bands;
- **the win/draw/loss model**: the probabilities that tomorrow is a win (return above +0.2%), a
  draw (within ±0.2%) or a loss (below −0.2%).

Each comes as a LightGBM, as a GRU and as a scikit-learn MLP. **LightGBM and the GRUs are
loaded from their latest saved versions, not retrained**, when this computer has them (an earlier
run saved them in `models/models`); only what is missing is trained. The MLP, the new method, is
trained, checked and saved, then compared with them. Bars, features, labels, folds, baselines,
metrics, the models and the sanity checks all come from the package, as the tests check them.

| Section | Question |
|---|---|
| 1 Bars, labels, features | What does a model see at the close of day t, and what must it forecast for t+1? Does any feature look ahead? |
| 2 Walk-forward | How are the folds cut, and what is purged at their boundaries? |
| 3 Baselines | What does each model have to beat? |
| 4 LightGBM | The two models as gradient-boosted trees, loaded from their latest saved version. |
| 5 GRU | The two models from a 60-day window, three input sets, loaded likewise. |
| 6 MLP | The two models as scikit-learn multilayer perceptrons, trained here, and a lead for tuning them. |
| 7 Results and sanity checks | On the synthetic tickers: W/D/L no better than the volatility baseline but better than the class frequencies; the point forecast no better than the EWMA median; the range better than a constant band. |
| 8 The checks can fail | A planted signal must be found; two leaks must be caught; a forecast must not change when later bars are removed. |
| 9 Save the final models | Each model trained in this run, trained once more on every labelled row and saved with its out-of-sample forecasts under a unique id. |
| 10 The MLP against the saved models | The MLP's out-of-sample losses against each saved LightGBM and GRU, per ticker, without retraining them. |

**Data.** The default is the six synthetic tickers (generated, offline, not market data). To run
on real prices, save them first with `save_all()` or the dashboard's **Download all**, then set
`DATA = 'local'` and list the symbols. **Do not commit this notebook with outputs**: run on saved
data, its tables and charts would be market data, and a test refuses a committed ML notebook
with outputs. Clear all outputs before committing.

**Needs** the `stats` and `ml` extras: `pip install -e ".[stats,ml]"` (for a CPU-only PyTorch,
first `pip install torch --index-url https://download.pytorch.org/whl/cpu`). With LightGBM and
the GRUs saved, a run takes about 5 minutes on four CPU cores, most of it the MLP, its controls and
its sweep. The first run trains and saves them all and takes about 13 minutes. `GRU_INPUTS` and
`GRU_TICKERS` limit what the GRUs run.

In [ ]:
# ── Environment setup: Google Colab vs local ────────────────────────
# Locally the project is installed once from the repo root with `pip install -e ".[stats,ml]"`,
# so `from src.tools...` resolves from any working directory. On Colab, an editable install is
# only seen by a new interpreter, so the repo root goes on sys.path for this kernel.
import os
import sys

REPO = '/content/drive/MyDrive/github/quant_dev'

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount('/content/drive')
    if not os.path.isdir(REPO):
        raise FileNotFoundError(f'No repo at {REPO}; edit REPO to point at your clone.')
    %pip install -q -e "{REPO}[stats,ml]"
    sys.path.insert(0, REPO)

In [ ]:
import os
import time
from functools import partial

import numpy as np
import pandas as pd
from IPython.display import display

from src.tools import ml_models as ml
from src.tools.price_return import synthetic_tickers

try:
    import lightgbm
    import sklearn
    import torch
except ImportError as exc:
    raise ImportError('This notebook needs the ml extra: pip install -e ".[stats,ml]"') from exc

print(f'LightGBM {lightgbm.__version__}, PyTorch {torch.__version__}, scikit-learn {sklearn.__version__}')

## Parameters

The design choices confirmed in the plan are the package's defaults: return quantiles at
`ml.TAUS`, a win/draw/loss threshold of 0.2%, an expanding walk-forward window of 756 rows
retrained every 63. `LONG` are the tickers the sanity checks judge: SYN-LEV starts in August
2022, which leaves too few test days after the 3-year warm-up, so it is shown but not judged.

In [ ]:
DATA = 'synthetic'                       # 'synthetic' (offline) or 'local' (saved Yahoo data)
TICKERS = synthetic_tickers() if DATA == 'synthetic' else ['ES=F', 'NQ=F', 'GC=F', 'CL=F']
LONG = [t for t in TICKERS if t != 'SYN-LEV']       # judged by the sanity checks
WDL_THRESHOLD = ml.WDL_THRESHOLD         # 0.002: a win above +0.2%, a loss below -0.2%
N_JOBS = None                            # tickers in parallel; None is one per CPU
GRU_TICKERS = TICKERS                    # the GRUs are the slow part; shorten this list to save time
GRU_INPUTS = list(ml.GRU_INPUTS)         # 'returns_range' (the default), 'returns', 'features'
GRU_STEP = 252                           # retrain every 252 rows; the plan's 63 takes four times as long
REUSE_SAVED = True                       # load the latest saved LightGBM and GRUs; train only what is missing
SAVE_MODELS = True                       # section 9 writes models/models (git-ignored) and model_parameters
MLP_SWEEP = True                         # section 6: a short regularization sweep on the first ticker

### Load the saved models, or train them

`models_for` gives a model's walk-forward forecasts for each ticker. With `REUSE_SAVED` it loads
the **latest saved version** (`ml.load_latest`): the newest one on this computer for the same
model, target, ticker, data source and threshold. Its settings or code may be older, which is
what makes it a version, and the table says when a version's code differs from today's. Only
tickers with nothing saved are trained, and they are saved in section 9. `VERSIONS` records which
version each model is.

In [ ]:
VERSIONS = []      # which version of each model this run uses
TRAINED = []       # (ticker, model, forecasts, step): trained in this run, saved in section 9
SAVED = {}         # (model, target, ticker): the saved model, loaded or (section 9) new


def models_for(name, target, make_model, tickers, step=63, reuse=None):
    """`name`'s walk-forward forecasts for each ticker: its latest saved version when reusing and
    one is saved here, else trained now with `make_model()` (and saved in section 9)."""
    reuse = REUSE_SAVED if reuse is None else reuse
    out = {}
    for t in (tickers if reuse else []):
        saved = ml.load_latest(name, target, t, source=DATA, wdl_threshold=WDL_THRESHOLD)
        if saved is not None:
            out[t], SAVED[(name, target, t)] = saved.forecasts, saved
            VERSIONS.append({'model': name, 'target': target, 'ticker': t, 'how': 'loaded',
                             'model_id': saved.model_id, 'created': saved.parameters['created'],
                             'code as today': saved.parameters['model']['code']
                             == ml.code_hash(type(saved.model))})
    missing = [t for t in tickers if t not in out]
    if missing:
        for fc in ml.evaluate_many(missing, make_model(), source=DATA, wdl_threshold=WDL_THRESHOLD,
                                   n_jobs=N_JOBS, step=step):
            out[fc.ticker] = fc
            TRAINED.append((fc.ticker, make_model(), fc, step))
            VERSIONS.append({'model': name, 'target': target, 'ticker': fc.ticker,
                             'how': 'trained now', 'model_id': None, 'created': None,
                             'code as today': True})
    return {t: out[t] for t in tickers}


def versions(name):
    v = pd.DataFrame(VERSIONS)
    return v[v['model'] == name].set_index(['target', 'ticker'])[['how', 'model_id', 'created',
                                                                  'code as today']]

## 1 Bars, labels, features

`C_t` is the close of bar t. The only forward-looking columns are the two labels: tomorrow's
return `y_ret = r_{t+1}` and tomorrow's outcome `y_wdl`: 1 (win) above +0.2%, 0 (draw) within
±0.2%, −1 (loss) below −0.2%. A return that spans a non-positive close (SYN-OIL on 2020-04-20) is
dropped, as `price_return` drops it.

The features (`ml.FEATURES`) use bar t and the bars before it, and none depends on the price
level. Two describe the recent wins and losses: `wdl`, today's outcome against the same
threshold, and `streak`, the run of wins (+) or losses (−) ending today, up to 10 either way,
reset to 0 by a draw.

In [ ]:
BARS = {t: ml.ticker_bars(t, source=DATA) for t in TICKERS}
DATA_BY_TICKER = {t: ml.dataset(b, WDL_THRESHOLD) for t, b in BARS.items()}
pd.DataFrame({t: {'rows': len(d), 'from': d.index[0].date(), 'to': d.index[-1].date(),
                  'wins': f"{(d['y_wdl'] == 1).mean():.1%}", 'draws': f"{(d['y_wdl'] == 0).mean():.1%}",
                  'losses': f"{(d['y_wdl'] == -1).mean():.1%}",
                  'longest streak': f"{d['streak'].min():+.0f} / {d['streak'].max():+.0f}"}
              for t, d in DATA_BY_TICKER.items()}).T

In [ ]:
DATA_BY_TICKER[TICKERS[0]][['ret_0', 'wdl', 'streak', 'y_ret', 'y_wdl']].tail(10)

**No look-ahead.** The test `ta_tools` uses for its indicators: cut the bars after t, recompute,
and every feature at or before t must be unchanged. A feature that peeked at a later bar would
change. (`tests/test_ml_models.py` runs this per feature; here it is shown once.)

In [ ]:
def check_no_lookahead(bars, cuts=(300, 1180, 2000)):
    full = ml.features(bars, WDL_THRESHOLD)
    for cut in cuts:
        pd.testing.assert_frame_equal(ml.features(bars.iloc[:cut + 1], WDL_THRESHOLD), full.iloc[:cut + 1],
                                      check_exact=False, rtol=1e-12)
    return f'{len(ml.FEATURES)} features unchanged at {len(cuts)} cut points'

check_no_lookahead(BARS[TICKERS[0]])

## 2 Walk-forward

Expanding window: the first fold trains on the first 756 rows and tests the next 63; each later
fold adds 63 rows to the training window. A training row whose label interval (t, t+h] reaches
the first test row is purged: h - 1 rows, so none for the next-day labels. Early stopping holds
out the last 20% of each training window, purged the same way.

In [ ]:
for h in (1, 5):
    train, test = ml.walk_forward(3000, horizon=h)[0]
    print(f'h = {h}: last training row {train[-1]}, its label reaches {train[-1] + h}, '
          f'first test row {test[0]}: {test[0] - train[-1] - 1} purged')
print(f'{TICKERS[0]}: {len(ml.walk_forward(len(DATA_BY_TICKER[TICKERS[0]])))} folds')

## 3 Baselines

Each is refitted per fold from the training rows, or uses only what is known at t.

**Win/draw/loss.** The training window's class frequencies; and a **volatility baseline**: the
chance that tomorrow clears the threshold given today's EWMA volatility (the training window's
standardized returns, scaled by today's volatility). With a fixed threshold, draws are commoner in
calm spells, so this baseline knows how likely a draw is, and nothing about direction.

**Return.** A constant-width band (the training quantiles), `price_range`'s lognormal ±zσ band on
the 250-day volatility, a 20-day σ band, and the EWMA volatility times the training quantiles of
`r_{t+1} / σ_t`. Each band's median is its point forecast.

In [ ]:
rows = []
for t in LONG:
    d = DATA_BY_TICKER[t]
    folds = ml.walk_forward(len(d))
    wdl = ml.wdl_baselines(d, folds)
    y = d.loc[wdl.index, 'y_wdl'].to_numpy()
    row = {'ticker': t}
    for b in ml.WDL_BASELINES:
        row[f'W/D/L log loss, {b}'] = ml.multiclass_log_loss(wdl[b].to_numpy(), y, ml.WDL_CLASSES).mean()
    bands = ml.range_baselines(d, folds)
    yr = d.loc[bands.index, 'y_ret']
    for band in ('constant', 'ewma_std_q'):
        s, _ = ml.range_scores({tau: bands[(band, tau)] for tau in ml.TAUS}, yr)
        row[f'pinball (bp), {band}'] = s['pinball'] * 1e4
        row[f'cover 68%, {band}'] = s['cover_68']
    rows.append(row)
pd.DataFrame(rows).set_index('ticker').round(4)

## 4 LightGBM

`ml.LightGBMWDLModel`: one `multiclass` model. `ml.LightGBMReturnModel`: one `quantile` model per
τ on `r_{t+1} / σ_t`, scaled back by σ_t; the median is the point forecast. Both: 15 leaves, at
least 50 rows a leaf, learning rate 0.03, up to 300 trees, early-stopped on the purged tail and
refitted on the whole training window, deterministic on one thread.

**Loaded, not retrained.** Each ticker's latest saved version is loaded with its out-of-sample
forecasts (`models_for`, above). Only a ticker with none saved on this computer is trained now,
and saved in section 9.

In [ ]:
t0 = time.time()
LGB_W = models_for('LightGBM', 'wdl', ml.LightGBMWDLModel, TICKERS)
LGB_R = models_for('LightGBM', 'return', ml.LightGBMReturnModel, TICKERS)
print(f'LightGBM, both models, {len(TICKERS)} tickers: {time.time() - t0:.0f} s')
display(versions('LightGBM'))
pd.DataFrame({t: {'folds': len(LGB_W[t].folds),
                  'W/D/L: median trees': int(LGB_W[t].folds['trees'].median()),
                  'W/D/L: top features': ', '.join(ml.feature_importance(LGB_W[t]).index[:3]),
                  'return median: top features': ', '.join(ml.feature_importance(LGB_R[t]).index[:3])}
              for t in TICKERS}).T

In [ ]:
t = TICKERS[0]
ml.plot_folds(LGB_W[t]).show()
ml.plot_feature_importance(ml.feature_importance(LGB_W[t]),
                           title=f'{t}: W/D/L model, share of gain per feature').show()

## 5 GRU

`ml.GRUWDLModel` and `ml.GRUReturnModel`: one GRU layer of 32 units over the last 60 rows of its
inputs, standardized with the training window's means and standard deviations. The W/D/L network
ends in three logits (cross-entropy); the return network in five quantiles of the standardized
return, scaled back by σ_t and **recalibrated**: each τ is shifted by the τ-quantile of the
network's own errors on the validation tail, which keeps its bands from being too narrow. Early
stopping on the purged tail, seeded per fold, deterministic on CPU.

**Three input sets, kept until a model is trained on real data** (the owner's choice):
`'returns_range'` (each day's return and its true range, the provisional default), `'returns'`
(the return alone) and `'features'` (all the features LightGBM reads). They retrain every
`GRU_STEP` rows, over the same test days as LightGBM. **Loaded, not retrained**, as LightGBM.

In [ ]:
GRU_W, GRU_R = {}, {}
t0 = time.time()
for inputs in GRU_INPUTS:
    name = f'GRU ({inputs})'
    GRU_W[inputs] = models_for(name, 'wdl', partial(ml.GRUWDLModel, features=inputs, threads=1),
                               GRU_TICKERS, step=GRU_STEP)
    GRU_R[inputs] = models_for(name, 'return', partial(ml.GRUReturnModel, features=inputs, threads=1),
                               GRU_TICKERS, step=GRU_STEP)
    print(f'{name}, both models: {time.time() - t0:.0f} s')
pd.concat({f'GRU ({i})': versions(f'GRU ({i})') for i in GRU_INPUTS})

## 6 MLP (scikit-learn)

The new method, always trained here. `ml.MLPWDLModel` and `ml.MLPReturnModel` at the owner's
starting point: **two hidden layers, each as wide as the feature list (18 neurons), ReLU, Adam**,
and scikit-learn's defaults for everything else (`alpha` 0.0001, batches of 200, learning rate
0.001, at most 200 epochs, no early stopping).

- **W/D/L:** an `MLPClassifier` with those settings, since a regressor gives no probabilities. It
  trains on all the training rows.
- **Return:** an `MLPRegressor` on `r_{t+1} / σ_t`, whose forecast is the centre. It trains on the
  training rows less the purged last 20%. Its errors on that held-out tail set each τ's offset,
  as the GRU's recalibration does. The median is the point forecast; the offsets give the bands.
- **Two departures from the defaults,** which the plan's rules need:
  - a fixed seed per fold, since the default draws a new one each run;
  - inputs standardized on the training rows, since the MLP does not scale them.

Forecasts are computed from the trained weights, one row at a time (the same numbers as
scikit-learn's `predict`), so a saved model needs only its weights.

In [ ]:
t0 = time.time()
MLP_W = models_for('MLP', 'wdl', ml.MLPWDLModel, TICKERS, reuse=False)
MLP_R = models_for('MLP', 'return', ml.MLPReturnModel, TICKERS, reuse=False)
print(f'MLP, both models, {len(TICKERS)} tickers: {time.time() - t0:.0f} s')
pd.DataFrame({t: {'folds': len(MLP_W[t].folds),
                  'W/D/L: epochs (median)': int(MLP_W[t].folds['n_iter'].median()),
                  'W/D/L: folds converged': f"{MLP_W[t].folds['converged'].mean():.0%}",
                  'return: epochs (median)': int(MLP_R[t].folds['n_iter'].median()),
                  'return: folds converged': f"{MLP_R[t].folds['converged'].mean():.0%}"}
              for t in TICKERS}).T

"Converged" is scikit-learn's own test: the training loss stopped improving by its tolerance
before the last epoch. At 200 epochs it rarely has, and it says so with a warning, recorded per
fold here rather than printed.

### A lead for tuning: regularization

At the starting configuration the networks fit the noise, and section 7 shows what that costs.
Here are the same models on the first ticker with more regularization (`alpha`), or with
scikit-learn's early stopping, passed through `params`. Above zero is better than the baseline.
Set `MLP_SWEEP = False` to skip this (about two minutes).

In [ ]:
if MLP_SWEEP:
    t0 = time.time()
    sweep, d = {}, DATA_BY_TICKER[TICKERS[0]]
    for label, params in [('starting point', {}), ('early_stopping=True', {'early_stopping': True}),
                          ('alpha=1', {'alpha': 1.0}), ('alpha=10', {'alpha': 10.0})]:
        w = ml.walk_forward_forecasts(d, ml.MLPWDLModel(params=params), ticker=TICKERS[0])
        r = ml.walk_forward_forecasts(d, ml.MLPReturnModel(params=params), ticker=TICKERS[0])
        rt = ml.range_table(r)
        sweep[label] = {'W/D/L log loss, gain over the volatility baseline': ml.wdl_scores(w)['gain_vs_volatility'],
                        'point MAE, gain over the EWMA median (bp)': ml.point_scores(r)['gain_vs_ewma'] * 1e4,
                        'range pinball, gain over the constant band':
                            1 - rt.loc['model', 'pinball'] / rt.loc['constant', 'pinball']}
    print(f'sweep: {time.time() - t0:.0f} s')
    display(pd.DataFrame(sweep).T.style.format({
        'W/D/L log loss, gain over the volatility baseline': '{:+.4f}',
        'point MAE, gain over the EWMA median (bp)': '{:+.2f}',
        'range pinball, gain over the constant band': '{:+.1%}'}).set_caption(f'{TICKERS[0]}: the MLP with more regularization'))

## 7 Results and sanity checks

On the synthetic tickers the sign of tomorrow's return cannot be forecast, but its volatility
can. So, for every method:

- **win/draw/loss**: the model must be no better than the volatility baseline (its 99% interval
  of the log-loss gain must not lie wholly above zero, and the gain must be at most 0.015), but
  it should beat the plain class frequencies, pooled over the tickers. The margin is there because
  the volatility baseline (EWMA) is not the best volatility forecast: the generator's own GARCH
  volatility beats it by up to 0.014 on SYN-INDEX, so a model may land a little above it;
- **the point forecast** (the median) must be no better than the EWMA band's median;
- **the range** must have a lower pinball loss than the constant band on each long ticker, misses
  that cluster less, and a pooled gain whose 95% interval lies wholly above zero.

LightGBM's and the GRUs' tables come from their saved forecasts, so they are the numbers of the
version loaded.

In [ ]:
WDL_COLS = ['days', 'share_draw', 'log_loss_frequencies', 'log_loss_volatility', 'log_loss_model',
            'gain_vs_frequencies', 'gain_vs_volatility', 'lower_vs_volatility', 'upper_vs_volatility',
            'no_better_than_volatility']


def results(name, wdl, ret):
    out = {}
    if wdl:
        w = pd.DataFrame([ml.wdl_scores(fc) for fc in wdl.values()])[WDL_COLS]
        display(w.style.format({c: '{:+.4f}' if c.startswith(('gain', 'lower', 'upper')) else '{:.4f}'
                                for c in WDL_COLS[2:-1]} | {'share_draw': '{:.1%}'})
                .set_caption(f'{name}: win/draw/loss'))
        _, pooled, above = ml.wdl_check([wdl[t] for t in LONG if t in wdl], 'frequencies', 'model')
        chance = bool(w.loc[w.index.isin(LONG), 'no_better_than_volatility'].all())
        print(f'{name}: W/D/L no better than the volatility baseline on every long ticker: {chance}; '
              f'pooled gain over the class frequencies {pooled["estimate"]:+.4f} '
              f'(95% interval {pooled["lower"]:+.4f} to {pooled["upper"]:+.4f})')
        out['W/D/L no better than volatility'] = chance
        out['W/D/L beats the frequencies'] = above
    if ret:
        p = pd.DataFrame([ml.point_scores(fc) for fc in ret.values()])
        display((p.drop(columns='no_better_than_ewma') * 1e4).round(3).assign(ok=p['no_better_than_ewma'])
                .style.set_caption(f'{name}: point forecast, mean absolute error (bp)'))
        ranges = pd.concat({t: ml.range_table(fc).loc[['constant', 'ewma_std_q', 'model'],
                                                     ['cover_68', 'cover_90', 'kupiec_p_68',
                                                      'christoffersen_p_68', 'cluster_ratio_68', 'pinball']]
                            for t, fc in ret.items()})
        ranges['pinball'] *= 1e4
        display(ranges.style.format({'cover_68': '{:.1%}', 'cover_90': '{:.1%}', 'kupiec_p_68': '{:.3f}',
                                     'christoffersen_p_68': '{:.2g}', 'cluster_ratio_68': '{:.2f}',
                                     'pinball': '{:.2f}'}).set_caption(f'{name}: range (pinball in bp)'))
        _, pooled, passes = ml.range_check([ret[t] for t in LONG if t in ret])
        point = bool(p.loc[p.index.isin(LONG), 'no_better_than_ewma'].all())
        print(f'{name}: point forecast no better than the EWMA median: {point}; range pooled gain '
              f'{pooled["estimate"]:.1%} (95% interval {pooled["lower"]:.1%} to {pooled["upper"]:.1%}); '
              f'range checks pass: {passes}')
        out['point no better than EWMA median'] = point
        out['range beats the constant band'] = passes
        out['range pooled gain'] = f'{pooled["estimate"]:.1%} ({pooled["lower"]:.1%} to {pooled["upper"]:.1%})'
    return out


SUMMARY = {'LightGBM': results('LightGBM', LGB_W, LGB_R)}

In [ ]:
for inputs in GRU_W:
    SUMMARY[f'GRU ({inputs})'] = results(f'GRU ({inputs})', GRU_W[inputs], GRU_R[inputs])

In [ ]:
SUMMARY['MLP'] = results('MLP', MLP_W, MLP_R)

### Charts

Reliability of the W/D/L forecasts (one curve per outcome; marker area grows with the days in
the bin), the return model's point forecast and bands through SYN-INDEX's scheduled sell-off, the
rolling coverage of its 68% interval, and every band's pinball loss against the constant one: for
LightGBM, then the same for the MLP.

In [ ]:
t = TICKERS[0]
ml.plot_wdl_reliability(LGB_W[t]).show()
ml.plot_wdl_reliability(LGB_W[t], source='volatility').show()
ml.plot_forecast_bands(LGB_R[t], BARS[t], '2020-02-01', '2020-04-30').show()
ml.plot_coverage(LGB_R[t]).show()
ml.plot_range_comparison([LGB_R[t] for t in LONG]).show()

In [ ]:
ml.plot_wdl_reliability(MLP_W[t]).show()
ml.plot_forecast_bands(MLP_R[t], BARS[t], '2020-02-01', '2020-04-30').show()
ml.plot_range_comparison([MLP_R[t] for t in LONG]).show()

## 8 The checks can fail

A check that cannot fail proves nothing; a model that always forecast the volatility baseline
would pass trivially. So controls on each W/D/L model trained in this run:

1. **A planted signal.** SYN-INDEX with each day's sign made to repeat the last 65% of the time
   (sizes and volatility kept). The W/D/L model must now beat the volatility baseline.
2. **A leaked future return.** `r_{t+1}` added as a feature. The check must fail.
3. **A centred window.** A 5-day average centred on t, a realistic look-ahead bug: it carries
   `r_{t+1}` and `r_{t+2}`. The check must fail.
4. **A forecast cannot see past its day.** Remove every bar after a test date (keeping the next
   close, which is only that date's label), refit that fold, and both models' forecasts must be
   identical.

The MLP gets all four. LightGBM and the GRUs loaded from saved versions passed theirs when they
were trained (an earlier run of this notebook, and the package's tests), so they are not
retrained here; any trained in this run get theirs.

In [ ]:
index_data = ml.dataset(ml.ticker_bars('SYN-INDEX'), WDL_THRESHOLD)
r = ml.bar_returns(ml.ticker_bars('SYN-INDEX'))
centred = index_data.assign(leak=r.rolling(5, center=True).mean().reindex(index_data.index)).dropna()
centred.attrs = index_data.attrs
planted = ml.dataset(ml.planted_signal_bars(), WDL_THRESHOLD)
CONTROL_COLS = ['days', 'log_loss_volatility', 'log_loss_model', 'gain_vs_volatility',
                'lower_vs_volatility', 'upper_vs_volatility', 'no_better_than_volatility',
                'beats_volatility']


def trained_now(prefix):
    return any(v['how'] == 'trained now' and v['model'].startswith(prefix) for v in VERSIONS)


def controls(make, leaky):
    runs = {'planted signal': ml.walk_forward_forecasts(planted, make()),
            'leaked r(t+1)': ml.walk_forward_forecasts(index_data.assign(leak=index_data['y_ret']), leaky),
            'centred window': ml.walk_forward_forecasts(centred, leaky)}
    out = pd.DataFrame({k: ml.wdl_scores(fc) for k, fc in runs.items()}).T[CONTROL_COLS]
    out['must'] = ['beat the volatility baseline', 'fail the check', 'fail the check']
    out['passes'] = [bool(out.loc['planted signal', 'beats_volatility']),
                     not out.loc['leaked r(t+1)', 'no_better_than_volatility'],
                     not out.loc['centred window', 'no_better_than_volatility']]
    return out


CONTROLS = {'MLP': controls(ml.MLPWDLModel, ml.MLPWDLModel(features=ml.FEATURES + ['leak']))}
if trained_now('LightGBM'):
    CONTROLS['LightGBM'] = controls(ml.LightGBMWDLModel,
                                    ml.LightGBMWDLModel(features=ml.FEATURES + ['leak'], max_trees=60))
pd.concat(CONTROLS)

In [ ]:
# The planted signal for each GRU input set trained in this run.
GRU_PLANTED = pd.DataFrame({
    f'GRU ({inputs})': ml.wdl_scores(ml.walk_forward_forecasts(
        planted, ml.GRUWDLModel(features=inputs), step=GRU_STEP))
    for inputs in GRU_INPUTS if trained_now(f'GRU ({inputs})')}).T
if len(GRU_PLANTED):
    display(GRU_PLANTED[['days', 'log_loss_volatility', 'log_loss_model', 'gain_vs_volatility',
                         'lower_vs_volatility', 'beats_volatility']])
else:
    print('Every GRU was loaded from a saved version: its planted-signal control ran when it was trained.')

In [ ]:
bars = ml.ticker_bars('SYN-INDEX')
checks = [('MLP', ml.MLPWDLModel, ml.MLPReturnModel, MLP_W, MLP_R)]
if trained_now('LightGBM'):
    checks.append(('LightGBM', ml.LightGBMWDLModel, ml.LightGBMReturnModel, LGB_W, LGB_R))
UNCHANGED = {}
for name, make_w, make_r, W, R in checks:
    if 'SYN-INDEX' not in W:
        continue                          # run on saved data: the check needs SYN-INDEX's forecasts
    for position in (900, 1700, 2400):
        date = index_data.index[position]
        d = ml.dataset(bars.iloc[:bars.index.get_loc(date) + 2], WDL_THRESHOLD)
        fold = next(f for f in ml.walk_forward(len(d)) if date in d.index[f[1]])
        same_w = ml.walk_forward_forecasts(d, make_w(), folds=[fold]).wdl.loc[date, 'model'].equals(
            W['SYN-INDEX'].wdl.loc[date, 'model'])
        same_r = ml.walk_forward_forecasts(d, make_r(), folds=[fold]).bands.loc[date, 'model'].equals(
            R['SYN-INDEX'].bands.loc[date, 'model'])
        UNCHANGED[(name, f'{date:%Y-%m-%d}')] = same_w and same_r
pd.Series(UNCHANGED, name='identical with the later bars removed', dtype=bool)

## Summary

At its starting configuration the MLP overfits the noise. So it passes the "no better than"
checks, which guard against look-ahead, by being worse than the baselines, and it fails the range
check and the planted signal. That is a configuration to tune, not a leak: section 6's sweep
shows every score moving towards the baselines as the regularization grows, and the leaks are
still caught.

In [ ]:
display(pd.DataFrame(SUMMARY))
for name, table in CONTROLS.items():
    print(f'Controls ({name} W/D/L): ' + '; '.join(
        f"{k}: {'passes' if ok else 'FAILS'}" for k, ok in table['passes'].items()))
for name, row in GRU_PLANTED.iterrows():
    print(f"{name}: planted signal found: {bool(row['beats_volatility'])} (gain {row['gain_vs_volatility']:+.4f})")
if UNCHANGED:
    print(f'Forecasts unchanged when later bars are removed: {all(UNCHANGED.values())}')

## 9 Save the final models

Every model trained in this run (the MLP, and any LightGBM or GRU that had no saved version) is
trained once more on every labelled row (`ml.finalize`, the same fit with all rows in the training
window) and saved:

- `models/models/<id>.pkl`, **git-ignored**: one object per model with everything in it: the
  parameters, the trained trees or weights, and the walk-forward out-of-sample forecasts. Those
  forecasts are what a later model is compared with, as section 10 does: a final model cannot be
  scored on the days it was trained on.
- `models/model_parameters/<id>.json`, **tracked**: the methodology (model, settings, features,
  threshold, walk-forward, a hash of the model code), the bars it was trained on (range and
  fingerprint), the final fit and summary scores. No per-day values, no weights.

**The id**, `{model}_{target}_{ticker}_{last bar}_{hash}` (e.g.
`mlp_wdl_SYN-INDEX_20260930_1c2d3e4f`): the hash covers the methodology and the bars, so the
same model on the same bars always gets the same id and is not saved twice, while a changed
setting, an edited model, or newer bars get a new id. `spec_id` hashes the methodology alone and
is shared by every ticker trained the same way. The loaded models are already saved; the table
lists the version of every model this run used.

In [ ]:
t0 = time.time()
NEW = []
for step in sorted({s for *_, s in TRAINED}):
    NEW += ml.finalize_many([(t, m, fc) for t, m, fc, s in TRAINED if s == step], n_jobs=N_JOBS,
                            source=DATA, wdl_threshold=WDL_THRESHOLD, step=step)
print(f'{len(NEW)} final models: {time.time() - t0:.0f} s')
for s in NEW:
    SAVED[(s.model.name, s.target, s.ticker)] = s
    if SAVE_MODELS:
        ml.save_model(s)
if SAVE_MODELS and NEW:
    print(f'saved in {ml.MODELS_DIR} and {ml.PARAMETERS_DIR}')
for v in VERSIONS:
    if v['how'] == 'trained now':
        s = SAVED[(v['model'], v['target'], v['ticker'])]
        v.update(model_id=s.model_id, created=s.parameters['created'])
pd.DataFrame(VERSIONS).set_index(['model', 'target', 'ticker'])

### Load a model back and forecast the next day

`ml.load_model(id)` returns the saved object, ready to forecast; `ml.load_latest(model, target,
ticker)` the newest saved version. `forecast_next()` applies it to the ticker's latest bar (the
one with no label yet), so after saving newer bars the same model forecasts the newer day without
retraining. `ml.list_models()` lists every saved model from the tracked parameters, and
`saved_here` shows which are on this computer.

In [ ]:
first = SAVED[('MLP', 'wdl', TICKERS[0])]
loaded = ml.load_model(first.model_id) if SAVE_MODELS else first
same = (loaded.forecasts.wdl.equals(first.forecasts.wdl)
        and loaded.forecast_next().equals(first.forecast_next()))
print(f'{loaded.model_id}: reloaded; forecasts identical to the trained model: {same}')

wdl_next, ret_next = {}, {}
for s in SAVED.values():
    f = s.forecast_next()
    key = (s.model.name, s.ticker)
    if s.target == 'wdl':
        wdl_next[key] = {'as of': f.name.date(), 'loss': f['loss'], 'draw': f['draw'], 'win': f['win']}
    else:
        close = BARS[s.ticker]['Close'].iloc[-1]
        ret_next[key] = {'as of': f.name.date(), 'close': close, 'median': close * (1 + f[0.5]),
                         '68% from': close * (1 + f[0.1587]), '68% to': close * (1 + f[0.8413])}
display(pd.DataFrame(wdl_next).T.sort_index().style.format({'loss': '{:.1%}', 'draw': '{:.1%}', 'win': '{:.1%}'})
        .set_caption('Next day: loss, draw and win probabilities'))
display(pd.DataFrame(ret_next).T.sort_index().style.format({c: '{:,.2f}' for c in ('close', 'median', '68% from', '68% to')})
        .set_caption('Next day: the close, its median forecast and the 68% range'))

## 10 The MLP against the saved LightGBM and GRUs

Nothing is retrained: the LightGBM and GRU forecasts are their saved versions' out-of-sample
forecasts (sections 4 and 5), the MLP's are this run's. On the days both forecast,
`ml.compare_forecasts(other, mlp)` gives the mean daily loss of each (log loss for win/draw/loss,
mean pinball for the return) and the gain of the MLP over the other model, with a 95% bootstrap
interval: `b_better` when it lies wholly above zero (the MLP is better), `a_better` when wholly
below. The chart shows the gain as a share of the other model's loss, so the two targets read on
the same footing; the table has every number.

In [ ]:
OTHERS = {'LightGBM': (LGB_W, LGB_R), **{f'GRU ({i})': (GRU_W[i], GRU_R[i]) for i in GRU_INPUTS}}
rows = []
for name, (W, R) in OTHERS.items():
    for target, theirs, ours in (('wdl', W, MLP_W), ('return', R, MLP_R)):
        for t in LONG:
            if t in theirs and t in ours:
                rows.append({**ml.compare_forecasts(theirs[t], ours[t]).to_dict(), 'ticker': t})
COMPARE = pd.DataFrame(rows).set_index(['target', 'model_a', 'ticker'])
display(COMPARE[['model_b', 'days', 'loss_a', 'loss_b', 'gain_b_over_a', 'lower', 'upper', 'b_better',
                 'a_better']].style.format({'loss_a': '{:.6f}', 'loss_b': '{:.6f}', 'gain_b_over_a': '{:+.6f}',
                                            'lower': '{:+.6f}', 'upper': '{:+.6f}'})
        .set_caption('The MLP (b) against each saved model (a), per ticker'))
ml.plot_model_comparison(COMPARE.reset_index()).show()
COMPARE.groupby(['target', 'model_a']).agg(tickers=('days', 'size'), MLP_better=('b_better', 'sum'),
                                           other_better=('a_better', 'sum'))